# Import 
- Prerequisites: 
  - Anaconda packages: `pandas, openpyxl`


## Result

Updated SPOD (MODEL_SOURCE) according to input Excel sheet 'DQM-Rules.xlsx' (DQR_EXPORT)

## Structure of this Notebook

1. Verif source and target
1. Import DQM Rules
1. Write new SPOD
1. Summary report of changes


## Configuration

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

MODEL_SOURCE = '/Users/bue/projects/geberit/DEAP/DB/IM_GEBERIT.json'
DQR_EXPORT = '/Users/bue/Documents/fyayc_local/Geberit/DQM/DQM-Rules.xlsx'

DESTINATION = 'merged.json'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path
import matplotlib.colors as mcolors
import seaborn as sns
from tqdm.autonotebook import tqdm

# openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.worksheet.table import Table
from openpyxl.utils.cell import get_column_letter
from openpyxl.comments import Comment
from openpyxl.styles import PatternFill

In [ ]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{configfile.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

# Load datasource (Excel)

In [ ]:
wb = load_workbook(DQR_EXPORT)
ws = wb.active
print(f"Loaded excel with tab '{ws.title}' containing {len(list(ws.rows))} rows")

## Verify structure

In [ ]:
for table in ws.tables.values():
    print(table)

In [ ]:
print(f"Found {len(ws.tables.values())} in worksheet") 

In [ ]:
# Verify that the expected columns are in place
# Blue
assert ws['A1'].value == 'DQM_Rule ID'
assert ws['B1'].value == 'Rulename'
assert ws['C1'].value == 'Ruledescription'

# Grey
assert ws['D1'].value == 'Binding Parameter'
assert ws['E1'].value == 'Rule Data Area'
assert ws['F1'].value == 'Rule Data System'
assert ws['G1'].value == 'Rule Data Source'
assert ws['H1'].value == 'Rule Data Field'
assert ws['I1'].value == 'Rule Data Exchange relevant'

#Blue
assert ws['J1'].value == 'Rule Quality Dimension'
assert ws['K1'].value == 'Rule Priority'
assert ws['L1'].value == 'Rule Type'
assert ws['M1'].value == 'Rule Status'
assert ws['N1'].value == 'Rulename_EN'
assert ws['O1'].value == 'Ruledescription_EN'

print("Expected columns found 😎")

# Data transformation

In [ ]:
def patch_system_name(dqr_system_name: str) -> str:
    if dqr_system_name == 'SAP C11':
        return 'DM_SAP'
    return dqr_system_name

In [ ]:
business_rules = []
missing_columns = {}

def unique_negative():
    num = -1
    while True:
        yield num
        num -= 1
        
sequence_generator = unique_negative()

In [ ]:
def find_table(spod: dict, table_name: str, context: tuple) -> (str, dict):
    for key, table in spod['tables'].items():
        if table['name'] in table_name:
            return (key, table)
    return None

def find_column(spod: dict, column_name: str, context: tuple) -> [tuple]:
    """Find column by name"""
    column_name_stripped = column_name.strip()
    result = []
    for key, column in spod['columns'].items():
        if column['database_col_id'] == column_name_stripped or column['name'] == column_name_stripped:
            result.append( (key, column, 1.0) )

    if len(result) == 0:
        new_column = missing_columns.get(column_name_stripped)
        if new_column is None:
            logging.warning(f"No suitable column for name '{column_name_stripped}' found on row {context[0].coordinate}: { [x.value for x in context] }")
            column = { 'name': column_name_stripped, 'id': next(sequence_generator) }
            systems = context[5]
            data_source = context[6]
            table = find_table(spod, data_source.value, context)
            if table is not None:
                column['table-id'] = table[0]
                column['database-name+'] = table[1]['name']
            else: # cannot resolve table
                column['database-name+'] = systems.value
            missing_columns[column_name_stripped] = column
            result.append( ( column['id'], column, -1 ) ) 
        else:
            result.append( (key, new_column, -.5) )
            
    return result

In [ ]:
def digest_row(spod: dict, record: tuple):
    cell_iterator = iter(record)
    rule_id = next(cell_iterator).value
    rule_name_de = next(cell_iterator).value
    rule_description_de = next(cell_iterator).value
    
    binding_parameter_cell = next(cell_iterator)
    rule_data_area = next(cell_iterator).value
    rule_data_system_cell = next(cell_iterator)

    # Table
    rule_data_source_cell = next(cell_iterator)

    # Field to check
    rule_data_field_cell = next(cell_iterator)

    
    # Process 'Binding Parameter'
    referenced_columns = []
    binding_parameter_cell_content = []
    not_found = []
    issue = False
    for colname in binding_parameter_cell.value.split(','):
        plain_name = colname.strip()
        res = find_column(spod, colname, record)
        assert len(res) > 0
        sure_hits = list(filter(lambda i: i[2] == 1, res))
        if len(sure_hits) < 1:
            binding_parameter_cell_content.append('!' + plain_name + '!')
            not_found.append(plain_name)
            issue = True
        else:
            binding_parameter_cell_content.append(plain_name)
        referenced_columns = referenced_columns + res
    
    new_val = ','.join(binding_parameter_cell_content)
    if issue:
        binding_parameter_cell.fill = PatternFill(fgColor="F5A9A9", fill_type = "solid")
        binding_parameter_cell.value = new_val
        binding_parameter_cell.comment = Comment(f"Cannot resolve columns: {', '.join(not_found)}", 'DQM Import.ipynb')
    
    # Process 'Rule Data Field'
    field_list_text = rule_data_field_cell.value
    
    field_list = field_list_text.split(' ')
    if '<br>' in field_list_text:
        field_list = field_list_text.split('<br>')
    
    checked_columns = []
    binding_parameter_cell_content = []
    
    issue = False
    for name in field_list:
        plain_name = name.strip()
        column_checked = find_column(spod, name, record)
        assert len(column_checked) > 0
        sure_hits_check = list(filter(lambda i: i[2] == 1, column_checked))     
        if len(sure_hits) < 1:
            binding_parameter_cell_content.append('!' + plain_name + '!')
            not_found.append(plain_name)
            issue = True
        else:
            binding_parameter_cell_content.append(plain_name)
        checked_columns = checked_columns + column_checked
    
    if issue:
        rule_data_field_cell.fill = PatternFill(fgColor="F5A9A9", fill_type = "solid")
        rule_data_field_cell.value = new_val
        rule_data_field_cell.comment = Comment(f"Cannot resolve columns: {', '.join(not_found)}", 'DQM Import.ipynb')
      
    
    print(f"Processing {rule_id} reference {referenced_columns}")

In [ ]:
digest_row(spod, next(ws.iter_rows(min_row=3, max_row=4, values_only=False)))

In [ ]:
for row in tqdm(ws.iter_rows(min_row=2, values_only=False), total=len(list(ws.rows)) - 1, unit=' row', dynamic_ncols=True):
    digest_row(spod, row)

In [ ]:
source_path = Path(DQR_EXPORT)
destination_path = Path(Path(DQR_EXPORT).parent, source_path.name + '.new.xlsx')
wb.save(destination_path)

# Visually verify

In [ ]:
import pandas

excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)